# Task 07 — MOGA Feasibility and Reproducibility Notebook

> **Notice & Computational Cost Warning**: MOGA (NSGA-II) optimization evaluates optics configurations over multiple generations. This notebook operates independently of the primary `bts.ipynb` workflow and enforces strict physical feasibility.

## 1. Setup & Package Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.nkm.moga import (
    BTSMOGAConfig,
    BTSMOGAProblem,
    run_bts_moga,
    save_moga_results_json
)

In [ ]:
# ── Simulation Configuration Summary ─────────────────────────────────────────
# Prints a table of all simulation parameters: their defaults (from config
# dataclasses) and any values reconfigured explicitly in this notebook.

import sys
from pathlib import Path

_repo_root = Path('..').resolve()
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from src.nkm.moga import BTSMOGAConfig

# ── Defaults (from class definition) ─────────────────────────────────────────
_default = BTSMOGAConfig()

# ── Values set in this notebook (Cell 4: config = BTSMOGAConfig(...)) ────────
_nb_pop_size = 20
_nb_n_gen    = 15
_nb_seed     = 42

# ── Table printer ─────────────────────────────────────────────────────────────
def _flag(default, notebook):
    return '⟵ reconfigured' if default != notebook else ''

rows = [
    # header
    ('Parameter', 'Default', 'This Notebook', 'Unit', 'Note'),
    ('-' * 35, '-' * 18, '-' * 18, '-' * 10, '-' * 20),
    # NSGA-II
    ('pop_size',
     _default.pop_size, _nb_pop_size, 'individuals',
     _flag(_default.pop_size, _nb_pop_size)),
    ('n_gen',
     _default.n_gen, _nb_n_gen, 'generations',
     _flag(_default.n_gen, _nb_n_gen)),
    ('random seed',
     _default.seed, _nb_seed, '—',
     _flag(_default.seed, _nb_seed)),
    # Decision variables
    ('n_quadrupoles',     9, 9, 'count', ''),
    ('quad_bounds (K)',
     f'[{_default.quad_bounds[0]}, {_default.quad_bounds[1]}]',
     f'[{_default.quad_bounds[0]}, {_default.quad_bounds[1]}]',
     'm⁻²', ''),
    # Physics constraints
    ('beta_max_limit',
     _default.beta_max_limit, _default.beta_max_limit, 'm', ''),
    ('mismatch_max_limit',
     _default.mismatch_max_limit, _default.mismatch_max_limit, '—', ''),
    ('aperture_radius',
     f'{_default.aperture_radius_m*1e3:.2f}',
     f'{_default.aperture_radius_m*1e3:.2f}', 'mm', ''),
    ('emittance_x',
     f'{_default.emittance_x_mrad:.0e}',
     f'{_default.emittance_x_mrad:.0e}', 'm·rad', ''),
    ('energy_spread (σ_δ)',
     f'{_default.energy_spread:.2e}',
     f'{_default.energy_spread:.2e}', '—', ''),
    # Objectives
    ('n_objectives',      3, 3, '—', 'f1=mismatch, f2=β_max, f3=disp'),
    ('feasibility_tol',   '1e-5', '1e-5', '—', 'constraint violation ≤ ε'),
    # Re-evaluation
    ('eval_n_mc_seeds',
     _default.eval_n_mc_seeds, _default.eval_n_mc_seeds, 'seeds', ''),
]

col_w = [36, 19, 19, 11, 28]
sep   = '+' + '+'.join('-' * w for w in col_w) + '+'
print()
print('  SIMULATION CONFIGURATION — 03_bts_moga_pareto')
print(sep)
for i, row in enumerate(rows):
    line = '|' + '|'.join(f' {str(v):<{col_w[j]-2}} ' for j, v in enumerate(row)) + '|'
    print(line)
    if i in (0, 1):
        print(sep)
print(sep)
print()


## 2. MOGA Configuration & Optimization Execution

In [ ]:
config = BTSMOGAConfig(
    pop_size=20,
    n_gen=15,
    seed=42
)
print(f"Running NSGA-II MOGA with pop_size={config.pop_size}, n_gen={config.n_gen}, seed={config.seed}...")
result = run_bts_moga(config)
print(f"Optimization completed in {result.runtime_seconds:.2f} seconds.")
print(f"Success: {result.success}, Feasible fraction: {result.feasible_fraction*100:.1f}%")
print(f"Feasible Pareto solutions found: {len(result.pareto_x)}")

## 3. Representative Pareto Solutions

In [ ]:
reps = result.representative_solutions
for name, sol in reps.items():
    print(f"=== {name.upper()} ===")
    print(f"  Total Mismatch (Mx+My): {sol['total_mismatch']:.4f}")
    print(f"  Peak Beta [m]:         {sol['peak_beta']:.4f}")
    print(f"  Residual Disp [m]:      {sol['residual_dispersion']:.4f}\n")

## 4. Visualization

### 4.1 Pareto Front Scatter Matrix (f1 vs f2, f1 vs f3, f2 vs f3)

In [ ]:
# Use Pareto front or fall back to least-infeasible population
pts  = result.pareto_f if len(result.pareto_f) > 0 else result.least_infeasible_f
ptsx = result.pareto_x if len(result.pareto_x) > 0 else result.least_infeasible_x
is_pareto = len(result.pareto_f) > 0
pop_label = 'Pareto Front' if is_pareto else 'Least-Infeasible Population'

f1 = pts[:, 0]   # Total mismatch Mx + My
f2 = pts[:, 1]   # Peak beta function (m)
f3 = pts[:, 2]   # Residual dispersion (m)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle(f'MOGA {pop_label}: Objective Scatter Matrix', fontsize=13, fontweight='bold')

pairs = [
    (f1, f2, f3, 'f₁: Mismatch (Mₓ+Mᵧ)', 'f₂: Peak β [m]', 'f₃: Residual Dispersion [m]'),
    (f1, f3, f2, 'f₁: Mismatch (Mₓ+Mᵧ)', 'f₃: Residual Dispersion [m]', 'f₂: Peak β [m]'),
    (f2, f3, f1, 'f₂: Peak β [m]', 'f₃: Residual Dispersion [m]', 'f₁: Mismatch (Mₓ+Mᵧ)'),
]

for ax, (xa, ya, ca, xl, yl, cl) in zip(axes, pairs):
    sc = ax.scatter(xa, ya, c=ca, cmap='plasma', s=40, edgecolors='k', linewidths=0.4, alpha=0.85)
    cb = plt.colorbar(sc, ax=ax, pad=0.02)
    cb.set_label(cl, fontsize=8)
    ax.set_xlabel(xl, fontsize=9)
    ax.set_ylabel(yl, fontsize=9)
    ax.grid(True, linestyle=':', alpha=0.5)

    # Mark representative solutions
    for sol_name, sol in reps.items():
        f1s = sol['total_mismatch']
        f2s = sol['peak_beta']
        f3s = sol['residual_dispersion']
        sol_vals = {'f1': f1s, 'f2': f2s, 'f3': f3s}
        x_key = 'f1' if xl.startswith('f₁') else ('f2' if xl.startswith('f₂') else 'f3')
        y_key = 'f1' if yl.startswith('f₁') else ('f2' if yl.startswith('f₂') else 'f3')
        xv = sol_vals.get(x_key, f1s)
        yv = sol_vals.get(y_key, f2s)
        ax.scatter(xv, yv, marker='*', s=200, color='red', zorder=5, edgecolors='black', linewidths=0.5)
        ax.annotate(sol_name, (xv, yv), textcoords='offset points', xytext=(5, 5),
                    fontsize=7, color='darkred')

plt.tight_layout()
plt.show()

### 4.2 Hypervolume Convergence History

In [ ]:
hv = result.history_hypervolume
if hv and len(hv) > 0:
    fig, ax = plt.subplots(figsize=(9, 4))
    gens = np.arange(1, len(hv) + 1)
    ax.plot(gens, hv, color='#2563EB', linewidth=2, marker='o', markersize=5)
    ax.fill_between(gens, hv, alpha=0.15, color='#2563EB')
    ax.set_xlabel('Generation', fontsize=11)
    ax.set_ylabel('Hypervolume Indicator', fontsize=11)
    ax.set_title('NSGA-II Hypervolume Convergence', fontsize=13, fontweight='bold')
    ax.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()
else:
    print("No hypervolume history available (feasible_fraction=0 or pop too small).")

### 4.3 Quadrupole Strength Profiles of Representative Solutions

In [ ]:
if reps and ptsx is not None and len(ptsx) > 0:
    n_quads = ptsx.shape[1]  # 9
    quad_labels = [f'Q{i+1}' for i in range(n_quads)]
    x_pos = np.arange(n_quads)
    width = 0.8 / max(len(reps), 1)

    COLORS_REP = ['#2563EB', '#D97706', '#059669', '#DC2626', '#7C3AED']

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
    ax.axhspan(-3, 0, alpha=0.05, color='blue', label='Defocusing region')
    ax.axhspan(0, 3, alpha=0.05, color='red', label='Focusing region')

    for i, (sol_name, sol) in enumerate(reps.items()):
        # Representative solution index — find closest Pareto point by objectives
        f1s = sol['total_mismatch']
        if is_pareto:
            dists = np.abs(pts[:, 0] - f1s)
            best_idx = int(np.argmin(dists))
            quad_k = ptsx[best_idx, :]
        else:
            dists = np.abs(pts[:, 0] - f1s)
            best_idx = int(np.argmin(dists))
            quad_k = ptsx[best_idx, :]

        offset = (i - len(reps)/2 + 0.5) * width
        bars = ax.bar(x_pos + offset, quad_k, width=width * 0.9,
                      label=sol_name, color=COLORS_REP[i % len(COLORS_REP)],
                      edgecolor='black', linewidth=0.5, alpha=0.85)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(quad_labels)
    ax.set_ylabel('Quadrupole Strength K [m⁻²]')
    ax.set_title('Representative Pareto Solutions: Quadrupole Strength Profiles', fontsize=13, fontweight='bold')
    ax.set_ylim(-3.3, 3.3)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(axis='y', linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()
else:
    print("No representative solution quad strengths available.")

### 4.4 Objective Trade-Off Radar Chart for Representative Solutions

In [ ]:
if reps:
    import matplotlib.patches as mpatches

    categories = ['Total\nMismatch', 'Peak β\n[m]', 'Residual\nDisp [m]']
    N = len(categories)
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]  # close the loop

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.set_title('Representative Solutions: Objective Trade-Offs (Radar)', fontsize=12, fontweight='bold', pad=20)

    # Collect all values for normalization
    all_mismatch = [s['total_mismatch'] for s in reps.values()]
    all_beta     = [s['peak_beta'] for s in reps.values()]
    all_disp     = [s['residual_dispersion'] for s in reps.values()]

    def normalize(vals):
        mn, mx = min(vals), max(vals)
        if mx == mn:
            return [0.5] * len(vals)
        return [(v - mn) / (mx - mn) for v in vals]

    norm_mismatch = normalize(all_mismatch)
    norm_beta     = normalize(all_beta)
    norm_disp     = normalize(all_disp)

    COLORS_REP = ['#2563EB', '#D97706', '#059669', '#DC2626', '#7C3AED']

    for i, (sol_name, sol) in enumerate(reps.items()):
        values = [norm_mismatch[i], norm_beta[i], norm_disp[i]]
        values += values[:1]
        ax.plot(angles, values, color=COLORS_REP[i % len(COLORS_REP)], linewidth=2, label=sol_name)
        ax.fill(angles, values, color=COLORS_REP[i % len(COLORS_REP)], alpha=0.15)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(['25%', '50%', '75%', '100%'], size=7)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print("No representative solutions to plot.")

## 5. Result Archival (JSON/CSV)

In [ ]:
output_dir = Path("results/publication_moga")
save_moga_results_json(result, output_dir=output_dir)
print(f"Saved MOGA results to {output_dir}/")